In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from src.ProcessKnowledgeGraph import ProcessKnowledgeGraph
from src.utils import *
from src.KnowledgeImporter import SimpleEventLogImporter, Keys

In [3]:
import importlib.resources

In [4]:
draw_graph(ProcessKnowledgeGraph())

GraphWidget(layout=Layout(height='800px', width='100%'))

# Load Event Log Information

In [5]:
from pandas.api.types import is_string_dtype, is_numeric_dtype, is_datetime64_any_dtype

In [6]:
import pandas as pd
import pm4py
log = pm4py.read_xes('../Sepsis Cases - Event Log.xes')


/home/lblinux/.local/lib/python3.10/site-packages/pm4py/util/dt_parsing/parser.py:77: UserWarning: ISO8601 strings are not fully supported with strpfromiso for Python versions below 3.11
  warnings.warn(


parsing log, completed traces ::   0%|          | 0/1050 [00:00<?, ?it/s]

### Extract Key Entities

In [7]:
from rdflib import XSD

In [8]:
# Manual annotation hints
entity_columns = []
value_columns = []
ignore_columns = [Keys.CASE, 'time:timestamp', 'lifecycle:transition']

In [9]:
pkg = ProcessKnowledgeGraph()
pkg.parse('./src/base_ontology.ttl')

importer = SimpleEventLogImporter(pkg, entity_columns=entity_columns, value_columns=value_columns, ignore_columns=ignore_columns)
importer.import_event_log_entities(log)
#draw_graph(importer.addition_graph)

InfectionSuspected, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema#boolean
org:group, object : ['A' 'B' 'C' 'D' 'E' 'F' 'G' 'H' '?' 'I']
=> Entity column
Added type owl class for org:group: http://example.org/type_org%3Agroup
DiagnosticBlood, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema#boolean
DisfuncOrg, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema#boolean
SIRSCritTachypnea, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema#boolean
Hypotensie, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema#boolean
SIRSCritHeartRate, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema#boolean
Infusion, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema#boolean
DiagnosticArtAstrup, object : [True nan False]
=> Value column of type http://www.w3.org/2001/XMLSchema

In [10]:
from itertools import zip_longest
draw_graph(importer.addition_graph, color_func=lambda _: dict(zip_longest(importer.addition_graph.all_nodes() - importer.pkg.all_nodes(), [], fillvalue='#99AA00')))

GraphWidget(layout=Layout(height='800px', width='100%'))

In [11]:
importer.load()
draw_graph(pkg)

GraphWidget(layout=Layout(height='800px', width='100%'))

In [12]:
from src.KnowledgeImporter import TextualImporter
x = TextualImporter(pkg)
# x.import_content_from_statement('The activity CRP describes that a blood test for measuring CRP values is performed')
x.import_content_from_statement('The process value CRP represents the mg of C-reactive protein per liter of blood in a blood test')
x.import_content_from_statement('The process value Lactic Acid measures the amount of lactic acid in a blood test')
x.import_content_from_statement('The process value Leucocytes represents the number of white blood cells in a blood test')
x.import_content_from_statement('The process value Hypoxie defines whether hypoxia has been detected for the patient')
x.visualize_addition_graph()

```turtle
log:processValue_CRP a :ProcessValue ;
    rdfs:comment "The process value CRP represents the mg of C-reactive protein per liter of blood in a blood test." .
```
```turtle
log:processValue_LacticAcid a :ProcessValue ;
    rdfs:label "Lactic Acid" ;
    rdfs:comment "Measures the amount of lactic acid in a blood test." .
```
```turtle
log:processValue_Leucocytes a :ProcessValue ;
    rdfs:comment "The number of white blood cells in a blood test." .
```
```turtle
log:processValue_Hypoxie a :ProcessValue ;
    rdfs:label "Hypoxie" ;
    :dataType xsd:boolean ;
    rdfs:comment "Defines whether hypoxia has been detected for the patient." .
```


GraphWidget(layout=Layout(height='620px', width='100%'))

In [13]:
x.ui.show_validation()

Output()

from src.KnowledgeImporter import TextualImporter
x = TextualImporter(pkg)
# x.import_content_from_statement('The activity CRP describes that a blood test for measuring CRP values is performed')
x.import_content_from_statement('The diagnose AA represents a patient with a tummy ache')
x.ui.show_validation()

from src.KnowledgeImporter import TextualImporter
x = TextualImporter(pkg)
x.import_content_from_statement('The diagnose AB represents a patient with a big nose')
x.ui.show_validation()

# Ontology Extraction

In [14]:
sepon = Graph()
sepon.bind('sepon', Namespace('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology'), override=True)
sepon.parse(importlib.resources.path('example_domains.sepsis', 'SEPON.ttl'))

<Graph identifier=N67669810365b447bb167e7f12b553d34 (<class 'rdflib.graph.Graph'>)>

In [15]:
query = '''
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX sepon: <http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#>

SELECT ?subject ?predicate ?object WHERE {
  {
    SELECT DISTINCT ?node WHERE {
      {
        ?node rdf:type sepon:Diagnostic_Biomarker .
      }
      UNION
      {
        ?node rdfs:subClassOf+ sepon:Diagnostic_Biomarker .
      }
    }
  }

  # Edges either going out from or coming into ?node
  {
    ?node ?predicate ?object .
    BIND(?node AS ?subject)
  }
  UNION
  {
    ?subject ?predicate ?node .
    BIND(?node AS ?object)
  }
}
'''

In [16]:
from src.KnowledgeImporter import ExistingOntologyImporter
ontology_importer = ExistingOntologyImporter(pkg)
ontology_importer.import_ontology(sepon, query)

Output()

In [470]:
len(ontology_importer.addition_graph)

1962

In [471]:
len(sepon)

12942

# ===============================================================

# Alignment

In [18]:
diagnoses = set(pkg.subjects(RDF.type, pkg.namespace_manager.expand_curie('log:type_Diagnose')))
diagnosis_filter = lambda node : node not in diagnoses
llm_approved = ontology_importer.align(target_node_filter=diagnosis_filter, addition_text_params={'blacklist' : {URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#reference')}})
llm_approved

Textualizing graphs for alignment...
Calculating basic alignment...
Trial by LLM...
ns1:C-Reactive_Protein -> log:processValue_CRP
ns1:Blood_Cell_Count -> log:processValue_Leucocytes
ns1:Leukocyte_Count -> log:processValue_Leucocytes
ns1:Blood_Chemistry_Measurement -> log:processValue_Leucocytes
ns1:Blood_Chemistry_Measurement -> log:processValue_CRP
ns1:Blood_Chemistry_Measurement -> log:processValue_LacticAcid
ns1:Lactate -> log:processValue_LacticAcid


{(rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Cell_Count'),
  rdflib.term.URIRef('http://example.org/processValue_Leucocytes')),
 (rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Chemistry_Measurement'),
  rdflib.term.URIRef('http://example.org/processValue_CRP')),
 (rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Chemistry_Measurement'),
  rdflib.term.URIRef('http://example.org/processValue_LacticAcid')),
 (rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Chemistry_Measurement'),
  rdflib.term.URIRef('http://example.org/processValue_Leucocytes')),
 (rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein'),
  rdflib.term.URIRef('http://example.org/processValue_CRP')),
 (rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOnt

In [43]:
ontology_importer.ui.show_alignment_validation(llm_approved)

Output()

In [44]:
g2 = Graph()
copy_namespaces(g2, ontology_importer.addition_graph)
for triple in ontology_importer.addition_graph:
    if URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Immunoprotein') in triple:
        g2.add(triple)
draw_graph(g2)

GraphWidget(layout=Layout(height='500px', width='100%'))

In [34]:
hidden = URIRef('http://example.org/hidden')
alignment_edges = list(filter(lambda triple: hidden not in triple, x))
for s,p,o in filter(lambda triple: OWL.sameAs in triple, x):
    print(s)

http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein
http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Chemistry_Measurement
http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate
http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Cell_Count
http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count
http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Chemistry_Measurement
http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Chemistry_Measurement


#### Textualizing

In [71]:
from rdflib import Graph, URIRef, Namespace, OWL, RDFS, RDF, Literal

In [107]:
from sentence_transformers import CrossEncoder, SentenceTransformer, util
from IPython.display import clear_output
import numpy as np

In [358]:
from src.utils import *

In [388]:
from rdflib.util import from_n3

In [540]:
addition_graph = ontology_importer.addition_graph
target_graph = pkg

diagnoses = set(pkg.subjects(RDF.type, pkg.namespace_manager.expand_curie('log:type_Diagnose')))
diagnosis_filter = lambda node : node not in diagnoses

alignment, addition_texts, target_texts = graph_alignment(addition_graph, target_graph, target_node_filter=diagnosis_filter, addition_text_params={'blacklist' : {URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#reference')}})

In [541]:
reverse_alignment, _, _ = graph_alignment(target_graph, addition_graph, addition_node_filter=diagnosis_filter, target_text_params={'blacklist' : {URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#reference')}})

In [542]:
pairs = dict()
reverse_pairs = dict()
for source_id, result in alignment.items():
    pairs.setdefault(source_id, set())
    ranks, top_ids = result
    for i, rank in enumerate(ranks):
        target_id = top_ids[i]
        reverse_pairs.setdefault(target_id, set())
        if source_id in reverse_alignment[target_id][1]:
            pairs[source_id].add(target_id)
            reverse_pairs[target_id].add(source_id)
            # print(f'{rank["score"]:.2f} {source_id.n3(addition_graph.namespace_manager)} {target_id.n3(target_graph.namespace_manager)}')
    # if ranks[0]['score'] > 0:

In [543]:
for source_id, matches in pairs.items():
        if len(matches) > 0:
            print(source_id.n3(addition_graph.namespace_manager))
            for target_id in matches:
                print(f'\t{target_id.n3(target_graph.namespace_manager)}')    

ns1:Heat_Shock_Protein
	log:org%3Agroup_U
ns1:Transport_Protein
	log:concept%3Aname_Admission%20IC
	log:concept%3Aname_Admission%20NC
	log:processValue_CRP
	log:concept%3Aname_IV%20Liquid
	log:concept%3Aname_LacticAcid
ns1:Procalcitonin
	log:type_Diagnose
ns1:Urokinase
	log:processValue_DiagnosticUrinaryCulture
	log:concept%3Aname_ER%20Sepsis%20Triage
ns1:High_Mobility_Group_Protein_B1
	log:org%3Agroup_N
	log:org%3Agroup_P
	log:org%3Agroup_I
	log:org%3Agroup_S
	log:org%3Agroup_B
	log:org%3Agroup_H
	log:org%3Agroup_M
	log:org%3Agroup_A
	log:org%3Agroup_J
ns1:Adult_Sepsis_Prognostic_Biomarker
	log:concept%3Aname_ER%20Sepsis%20Triage
ns1:Peptidase
	:Resource
	xsd:boolean
	log:concept%3Aname_Release%20D
	log:concept%3Aname_Release%20E
	log:concept%3Aname_CRP
	log:concept%3Aname_ER%20Triage
ns1:Prognostic_Biomarker
	log:processValue_DiagnosticUrinarySediment
	log:processValue_DiagnosticBlood
ns1:Acute_Physiology_and_Chronic_Health_Evaluation_II_Clinical_Classification
	log:concept%3Aname_ER

In [520]:
for target_id, matches in reverse_pairs.items():
        if len(matches) > 0:
            print(target_id.n3(target_graph.namespace_manager))
            for source_id in matches:
                print(f'\t{source_id.n3(addition_graph.namespace_manager)}')    

log:org%3Agroup_N
	ns1:Granulocyte_Count
	ns1:N-Terminal_Fragment_Brain_Natriuretic_Protein
	ns1:Protein_Family
	ns1:Tumor_Necrosis_Factor
log:concept%3Aname_ER%20Sepsis%20Triage
	ns1:Fight_Action_to_Neutralize_Sepsis_Score
	ns1:Acute_Physiology_and_Chronic_Health_Evaluation_II_Clinical_Classification
	ns1:Adult_Sepsis_Prognostic_Biomarker
	ns1:Adult_Sepsis_Diagnostic_Biomarker
	ns1:1107116
log:org%3Agroup_R
	ns1:Protein_Family
	ns1:Tumor_Necrosis_Factor
log:org%3Agroup_M
	ns1:MHC_Class_II_Protein
	ns1:Apolipoprotein_M
log:org%3Agroup_S
	ns1:High_Mobility_Group_Protein_B1
	ns1:Protein_S100-A9
	ns1:Protein_S100-A8
log:org%3Agroup_U
	ns1:Heat_Shock_Protein
	ns1:Protein_Family
log:processValue_CRP
	ns1:C-reactive_Protein_to_Albumin_Ratio
	ns1:C-Reactive_Protein
	ns1:C_Reactive_Protein_to_Mean_Platelet_Volume_Ratio
	ns1:Blood_Cell_Count
	ns1:Blood_Chemistry_Measurement
log:org%3Agroup_H
	ns1:HLA-DR_Antigen
	ns1:Histone_H3
	ns1:Haptoglobin
log:concept%3Aname_Release%20B
	ns1:GTPase
	ns1:lnc

In [483]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from dotenv import load_dotenv
load_dotenv()


llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")
prompt = ChatPromptTemplate.from_messages(
    [
        (
        "system",
'''You are a knowledge importer for a knowledge-graph-based business process management system.
Your task is to decide whether two stringifications of knowledge graph are related to the same concept.
Please output only True or False, whether the entities represented are related.''',
        ),
        ("human", "String 1:\n{str1}\n\n\nString 2:\n{str2}"),
    ]
)
chain = prompt | llm

def trial_by_llm(source_id, target_id):
    #print(prompt.format(str1=addition_texts[source_id], str2=target_texts[target_id]))
    return chain.invoke(
        {
            "str1": addition_texts[source_id],
            "str2": target_texts[target_id],
        }
    ).content == 'True'

trial_by_llm(URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein'), URIRef('http://example.org/processValue_CRP'))

True

In [531]:
for i in range(0, 10):
    print(f'{i}/{len(addition_texts)}', end='\r')
    if i == 5:
        print('Hello!')

Hello!


In [547]:
llm_approved = set()
# for source_id, matches in pairs.items():
for index, source_id in enumerate(pairs.keys()):#(alignment.keys()):
        print(f'{index}/{len(alignment)}', end='\r')
        matches = pairs[source_id]
        if len(matches) > 0:
            # print(source_id.n3(addition_graph.namespace_manager))
            for target_id in matches:
                # print(f'\t{target_id.n3(target_graph.namespace_manager)}')    
                response = trial_by_llm(source_id, target_id)
                if response:
                    llm_approved.add((source_id, target_id))
                    print(f'{source_id.n3(addition_graph.namespace_manager)} -> {target_id.n3(target_graph.namespace_manager)}')

ns1:Lactate -> log:processValue_LacticAcid
ns1:Blood_Chemistry_Measurement -> log:processValue_LacticAcid
ns1:Blood_Chemistry_Measurement -> log:processValue_Leucocytes
ns1:Blood_Cell_Count -> log:processValue_Leucocytes
ns1:C_Reactive_Protein_to_Mean_Platelet_Volume_Ratio -> log:processValue_CRP
ns1:Leukocyte_Count -> log:processValue_Leucocytes
ns1:C-Reactive_Protein -> log:processValue_CRP


In [509]:
g = Graph()
copy_namespaces(g, ontology_importer.addition_graph)
colors = dict()
for source_id, target_id in llm_approved:
    g.add((source_id, URIRef('isSameAs'), target_id))
    colors[source_id] = '#99AA00'
    colors[target_id] = '#1100AA' 

draw_graph(g, lambda _ : colors)

GraphWidget(layout=Layout(height='670px', width='100%'))

In [537]:
_matches = [
 (URIRef('http://example.org/processValue_CRP'),         URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein')),
 (URIRef('http://example.org/processValue_LacticAcid'),  URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate')),
 (URIRef('http://example.org/processValue_Leucocytes'),  URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count')),
 # (URIRef('http://example.org/processValue_Hypoxie'),     URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia')) Not a biomarker
]

In [501]:
list(filter(lambda x : 'CRP' in x, reverse_pairs))

[rdflib.term.URIRef('http://example.org/processValue_CRP'),
 rdflib.term.URIRef('http://example.org/concept%3Aname_CRP')]

In [544]:
for x,y in _matches:
    printmd(f'### {x} -> {y}')
    print(pairs[y])
    print(reverse_pairs[x])

### http://example.org/processValue_CRP -> http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein

{rdflib.term.URIRef('http://example.org/processValue_CRP')}
{rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-reactive_Protein_to_Albumin_Ratio'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C_Reactive_Protein_to_Mean_Platelet_Volume_Ratio'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Coagulation_Factor'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Chemistry_Test'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Chemistry_Measurement'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Cell_Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#S100_Calcium_Binding_Protein'),

### http://example.org/processValue_LacticAcid -> http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate

{rdflib.term.URIRef('http://example.org/type_Diagnose'), rdflib.term.URIRef('http://example.org/processValue_LacticAcid'), rdflib.term.URIRef('http://example.org/concept%3Aname_LacticAcid')}
{rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Uric_Acid'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate_to_Pyruvate_Ratio'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Nutrition_Assessment'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Chemistry_Test'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Cell_Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Serum_Chemistry_Measurement'), rdflib.term.URIRef('http://www.semanticweb.org

### http://example.org/processValue_Leucocytes -> http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count

{rdflib.term.URIRef('http://example.org/processValue_Leucocytes'), rdflib.term.URIRef('http://example.org/org%3Agroup_H'), rdflib.term.URIRef('http://example.org/org%3Agroup_B'), rdflib.term.URIRef('http://example.org/org%3Agroup_K'), rdflib.term.URIRef('http://example.org/concept%3Aname_Leucocytes'), rdflib.term.URIRef('http://example.org/type_org%3Agroup'), rdflib.term.URIRef('http://example.org/org%3Agroup_C')}
{rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Monocyte_Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Neutrophil_Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Eosinophil_Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Cell_Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count'), rdflib.term.URIRef('http://www.semanticweb.o

In [545]:
for x,y in _matches:
    printmd(f'### {x} -> {y}')
    print(alignment[y][1])
    print(reverse_alignment[x][1])

### http://example.org/processValue_CRP -> http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein

[rdflib.term.URIRef('http://example.org/processValue_CRP'), rdflib.term.URIRef('http://example.org/concept%3Aname_ER%20Sepsis%20Triage'), rdflib.term.URIRef('http://example.org/processValue_DiagnosticUrinarySediment'), rdflib.term.URIRef('http://example.org/processValue_DiagnosticOther'), rdflib.term.URIRef('http://example.org/processValue_DiagnosticXthorax'), rdflib.term.URIRef('http://example.org/processValue_DiagnosticLacticAcid'), rdflib.term.URIRef('http://example.org/processValue_DiagnosticIC'), rdflib.term.URIRef('http://example.org/processValue_DiagnosticSputum'), rdflib.term.URIRef('http://example.org/processValue_DiagnosticLiquor'), rdflib.term.URIRef('http://example.org/processValue_Leucocytes')]
[rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C_Reactive_Protein_to_Mean_Platelet_Volume_Ratio'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-reactive_Protein_to_Albumin_Ratio'), rdflib.term.URI

### http://example.org/processValue_LacticAcid -> http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate

[rdflib.term.URIRef('http://example.org/concept%3Aname_ER%20Sepsis%20Triage'), rdflib.term.URIRef('http://example.org/type_Diagnose'), rdflib.term.URIRef('http://example.org/concept%3Aname_LacticAcid'), rdflib.term.URIRef('http://example.org/processValue_LacticAcid'), rdflib.term.URIRef('http://example.org/org%3Agroup_P'), rdflib.term.URIRef('http://example.org/org%3Agroup_F'), rdflib.term.URIRef('http://example.org/org%3Agroup_L'), rdflib.term.URIRef('http://example.org/processValue_DiagnosticUrinarySediment'), rdflib.term.URIRef('http://example.org/org%3Agroup_B'), rdflib.term.URIRef('http://example.org/org%3Agroup_A')]
[rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Chemistry_Measurement'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate_to_Pyruvate_Ratio'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Chemistry_Test'), rdflib.term.URIRef('http://

### http://example.org/processValue_Leucocytes -> http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count

[rdflib.term.URIRef('http://example.org/type_org%3Agroup'), rdflib.term.URIRef('http://example.org/concept%3Aname_ER%20Sepsis%20Triage'), rdflib.term.URIRef('http://example.org/type_Diagnose'), rdflib.term.URIRef('http://example.org/concept%3Aname_Leucocytes'), rdflib.term.URIRef('http://example.org/org%3Agroup_K'), rdflib.term.URIRef('http://example.org/org%3Agroup_H'), rdflib.term.URIRef('http://example.org/processValue_Leucocytes'), rdflib.term.URIRef('http://example.org/org%3Agroup_C'), rdflib.term.URIRef('http://example.org/org%3Agroup_B'), rdflib.term.URIRef('http://example.org/concept%3Aname_IV%20Antibiotics')]
[rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Cell_Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Cell_Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/on

In [383]:
for source_id, result in reverse_alignment.items():
    ranks, top_ids = result
    print(f'{ranks[0]["score"]:.2f} {source_id.n3(pkg.namespace_manager)} {top_ids[0].n3(sepon.namespace_manager)}')

-11.03 log:processValue_SIRSCriteria2OrMore :Oxidoreductase
-5.50 log:Diagnose_UC :Complement_Receptor_Type_1
-10.41 log:concept%3Aname_Release%20C :C-C_Motif_Chemokine_2
-10.16 log:org%3Agroup_B :High_Mobility_Group_Protein_B1
-4.87 log:Diagnose_ED :Complement_Receptor_Type_1
-4.36 log:Diagnose_GC :Complement_Receptor_Type_1
-3.86 log:Diagnose_AD :ADM
-4.05 log:Diagnose_MD :Complement_Receptor_Type_1
-4.59 log:Diagnose_BB :Complement_Receptor_Type_1
-3.62 log:Diagnose_H :Complement_Receptor_Type_1
-4.81 log:Diagnose_DC :Complement_Receptor_Type_1
-4.22 log:Diagnose_YB :Complement_Receptor_Type_1
-9.24 log:processValue_DiagnosticUrinarySediment :Diagnostic_Criteria
-8.90 log:processValue_DiagnosticLiquor :Diagnostic_Criteria
-5.49 log:Diagnose_MA :Complement_Receptor_Type_1
-3.76 log:Diagnose_NA :Complement_Receptor_Type_1
-3.17 log:Diagnose_TC :Complement_Receptor_Type_1
-10.76 log:concept%3Aname_Release%20B :lncRNA
-10.02 log:org%3Agroup_K :Vitamin_K-Dependent_Protein_C
-10.23 log:or

In [351]:
# TODO
addition_graph = ontology_importer.addition_graph
target_graph = pkg
# =========
addition_texts = textualize_graph(addition_graph, graph_annotations_properties(sepon, ))
target_texts = textualize_graph(target_graph)

In [352]:
%%time
bi_encoder = SentenceTransformer("all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Prepare embeddings
source_ids, source_texts_list = zip(*addition_texts.items())
target_ids, target_texts_list = zip(*target_texts.items())

target_embeddings = bi_encoder.encode(target_texts_list, convert_to_tensor=True)
source_embeddings = bi_encoder.encode(source_texts_list, convert_to_tensor=True)

# Initial retrieval using cosine similarity
cosine_scores = util.cos_sim(source_embeddings, target_embeddings)

CPU times: user 1.22 s, sys: 250 ms, total: 1.47 s
Wall time: 4.72 s


In [353]:
def top_k_nodes(i, top_k=5, top_k_outer=20):
    source_id = source_ids[i]
    top_scores, top_indices = cosine_scores[i].topk(top_k_outer, sorted=True)
    top_values = [target_texts_list[index] for index in top_indices]
    ranks = cross_encoder.rank(source_texts_list[i], top_values, top_k=top_k, return_documents=True)

    # The following holds:
    #for i, rank in enumerate(ranks):
    #    assert target_texts_list[top_indices[rank['corpus_id']]] == rank['text']
    
    indices_in_collection = [top_indices[rank['corpus_id']].item() for rank in ranks]

    return ranks, indices_in_collection

In [354]:
%%time
# Rerank top-N candidates for each source document
top_k = 20  # number of top candidates to rerank
results = {}

for i, source_id in enumerate(source_ids):
    print(f'{i}/{len(addition_texts)}', end='\r')
    ranks, indices_in_collection = top_k_nodes(i)
    results[source_id] = ranks, indices_in_collection

CPU times: user 10.6 s, sys: 1.75 s, total: 12.4 s
Wall time: 8.65 s


In [355]:
for source_id, result in results.items():
    ranks, indices_in_collection = result
    if ranks[0]['score'] > 0:
        print(f'{ranks[0]["score"]:.2f} {sepon.namespace_manager.qname(source_id)} {target_ids[indices_in_collection[0]].n3(pkg.namespace_manager)}')

0.59 Neonatal_Sepsis_Biomarker log:concept%3Aname_ER%20Sepsis%20Triage
1.69 ns202:Creatinine_Ratio_plus_SOFA_Score _:n3cb85415b3474c7c88fa92676f2c179db1
2.60 owl:Class owl:Class
0.31 NKG2-D_Type_II_Integral_Membrane_Protein log:concept%3Aname_ER%20Sepsis%20Triage
0.13 Neutrophil_to_Lymphocyte_Ratio_Measurement log:concept%3Aname_ER%20Sepsis%20Triage
0.93 Maximum_Aggregation_Rate log:type_org%3Agroup
0.76 Red_Blood_Cell_Distribution_Width_to_Platelet_Ratio owl:Class
0.73 C-reactive_Protein_to_Albumin_Ratio _:n3cb85415b3474c7c88fa92676f2c179db1
0.40 Paediatric_Sepsis_Diagnostic_Biomarker log:type_Diagnose
1.95 C-Reactive_Protein log:processValue_CRP


In [305]:
results[list(results.keys())[15]][0][0]['score']

np.float32(-1.9264717)

In [453]:
def textualize_node(node, graph, annotation_properties):

    def strip_uri(uri):
        try:
            return graph.namespace_manager.qname(uri).split(':')[-1]
        except:
            return str(uri)
    
    annotation_facts = list(filter(lambda triple: triple[1] in annotation_properties, graph.triples((node, None, None)))) # Should be graph.triples((node, annotation_path, None)), but returned triples have weird path
    x = [print(y) for y in annotation_facts]
    context = ''
    label = next(filter(lambda triple: triple[1] == RDFS.label, annotation_facts), None)
    context = str(label[2]) if label else strip_uri(node)
    clazz = next(filter(lambda triple: triple[1] == RDF.type, annotation_facts), None)
    context += f' a {strip_uri(clazz[2])}' if clazz else ''
    for s,p,o in annotation_facts:
        if p not in [RDFS.label, RDF.type]:
            o_str = strip_uri(o).replace('\n', ' ')
            context += f'\n{strip_uri(p)}: {o_str}'
            
    return node, context

print('\n\n')
print(textualize_node(URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein'), graph=addition_graph, annotation_properties=graph_annotations_properties(addition_graph))[1])




(rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Adult_Sepsis_Diagnostic_Biomarker'))
(rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Neonatal_Sepsis_Biomarker'))
(rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein'), rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Paediatric_Sepsis_Diagnostic_Biomarker'))
(rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/Sepsi

dict_keys([rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#CD177_Gene'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Butorphanol'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#THBD_Gene'), rdflib.term.URIRef('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C32828'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Has_Prognostic_Biomarker'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Serine/Threonine_Protein_Kinase'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Amprenavir'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Cefixime'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#FCGR1A_Gene'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/onto

In [525]:
sorted(source_ids)

[rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#(-)-Methyl_Jasmonate'),
 rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#1107116'),
 rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#12-oxo-trans-10-dodecenoic_acid'),
 rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#20S_Proteasome'),
 rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#ADM'),
 rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Acute_Physiology_and_Chronic_Health_Evaluation_II_Clinical_Classification'),
 rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Adenosine_Diphosphate'),
 rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Adult_Sepsis_Diagnostic_Biomarker'),
 rdflib.term.URIRef('http://www.semanticweb.org/zchero

In [521]:
for x,y in _matches:
    printmd(f'### {x} -> {y}')
    for val in top_k_nodes(source_ids.index(y), top_k=5)[0]:
        print(val)

### http://example.org/processValue_CRP -> http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein

{'corpus_id': 0, 'score': np.float32(1.9467916), 'text': 'processValue_CRP a ProcessValue\ncomment: The mg of C-reactive protein per liter of blood in a blood test.'}
{'corpus_id': 1, 'score': np.float32(1.4531388), 'text': 'ER Sepsis Triage a Activity'}
{'corpus_id': 8, 'score': np.float32(1.0217602), 'text': 'RC a type_Diagnose'}
{'corpus_id': 2, 'score': np.float32(0.98055553), 'text': 'CC a type_Diagnose'}
{'corpus_id': 19, 'score': np.float32(0.9474606), 'text': 'C a type_Diagnose'}


### http://example.org/processValue_LacticAcid -> http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate

{'corpus_id': 7, 'score': np.float32(-2.637757), 'text': 'ER Sepsis Triage a Activity'}
{'corpus_id': 11, 'score': np.float32(-3.9368927), 'text': 'P a type_Diagnose'}
{'corpus_id': 1, 'score': np.float32(-3.9820943), 'text': 'LacticAcid a Activity'}
{'corpus_id': 2, 'score': np.float32(-4.0096974), 'text': 'HC a type_Diagnose'}
{'corpus_id': 9, 'score': np.float32(-4.0469728), 'text': 'CC a type_Diagnose'}


### http://example.org/processValue_Leucocytes -> http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count

{'corpus_id': 18, 'score': np.float32(-3.1558256), 'text': 'ER Sepsis Triage a Activity'}
{'corpus_id': 8, 'score': np.float32(-4.7146673), 'text': 'LE a type_Diagnose'}
{'corpus_id': 2, 'score': np.float32(-4.775344), 'text': 'Leucocytes a Activity'}
{'corpus_id': 0, 'score': np.float32(-4.903934), 'text': 'processValue_Leucocytes a ProcessValue\ncomment: The number of white blood cells in a blood test.'}
{'corpus_id': 9, 'score': np.float32(-5.1802225), 'text': 'OC a type_Diagnose'}


### http://example.org/processValue_Hypoxie -> http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia

ValueError: tuple.index(x): x not in tuple

In [279]:
x = bi_encoder.encode_document(target_texts_list[target_ids.index(URIRef('http://example.org/Diagnose_RC'))], convert_to_tensor=True)
y = bi_encoder.encode_document(target_texts_list[target_ids.index(URIRef('http://example.org/processValue_LacticAcid'))], convert_to_tensor=True)
z = bi_encoder.encode_document(source_texts_list[source_ids.index(URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate'))], convert_to_tensor=True)

(bi_encoder.similarity(x,z), bi_encoder.similarity(y,z))

(tensor([[0.2351]], device='cuda:0'), tensor([[0.5998]], device='cuda:0'))

In [280]:
x = target_embeddings[target_ids.index(URIRef('http://example.org/Diagnose_RC'))]
y = target_embeddings[target_ids.index(URIRef('http://example.org/processValue_LacticAcid'))]
z = source_embeddings[source_ids.index(URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate'))]

(bi_encoder.similarity(x,z), bi_encoder.similarity(y,z))

(tensor([[0.2351]], device='cuda:0'), tensor([[0.5998]], device='cuda:0'))

# Ontology Extraction

In [274]:
print(target_texts_list[target_ids.index(URIRef('http://example.org/Diagnose_RC'))])

RC a type_Diagnose


In [272]:
print(target_texts_list[target_ids.index(URIRef('http://example.org/processValue_LacticAcid'))])

Lactic Acid a ProcessValue
comment: Measures the amount of lactic acid in a blood test.


In [271]:
print(source_texts_list[source_ids.index(URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate'))])

Lactate a Class
subClassOf: Adult_Sepsis_Diagnostic_Biomarker
subClassOf: Adult_Sepsis_Prognostic_Biomarker
subClassOf: Paediatric_Sepsis_Diagnostic_Biomarker
subClassOf: Paediatric_Sepsis_Prognostic_Biomarker
Synonyms: Lactate (substance) LACTIC ACID, UNSPECIFIED FORM lactic acid Lactic Acid Lactate
definition: A hydroxy monocarboxylic acid anion that is the conjugate base of lactic acid, arising from deprotonation of the carboxy group.


In [168]:
url = URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#' + 'C-Reactive_Protein')
top_k_nodes(source_ids.index(url), top_k=20)[0]

[{'corpus_id': 4,
  'score': np.float32(-3.0191083),
  'text': 'log:concept%3Aname_Leucocytes a <http://infs.cit.tum.de/karibdis/baseontologyActivity> ;\n    rdfs:label "Leucocytes"'},
 {'corpus_id': 2,
  'score': np.float32(-3.039882),
  'text': 'log:concept%3Aname_IV%20Antibiotics a <http://infs.cit.tum.de/karibdis/baseontologyActivity> ;\n    rdfs:label "IV Antibiotics"'},
 {'corpus_id': 6,
  'score': np.float32(-3.1868439),
  'text': 'log:concept%3Aname_CRP a <http://infs.cit.tum.de/karibdis/baseontologyActivity> ;\n    rdfs:label "CRP"'},
 {'corpus_id': 0,
  'score': np.float32(-3.2420135),
  'text': 'log:concept%3Aname_ER%20Sepsis%20Triage a <http://infs.cit.tum.de/karibdis/baseontologyActivity> ;\n    rdfs:label "ER Sepsis Triage"'},
 {'corpus_id': 12,
  'score': np.float32(-3.2995138),
  'text': 'log:Diagnose_RC a log:type_Diagnose ;\n    rdfs:label "RC"'},
 {'corpus_id': 19,
  'score': np.float32(-3.3122137),
  'text': 'log:Diagnose_YD a log:type_Diagnose ;\n    rdfs:label "YD

In [132]:
results[URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein')]

[{'corpus_id': 3,
  'score': np.float32(-3.019106),
  'text': 'log:concept%3Aname_Leucocytes a <http://infs.cit.tum.de/karibdis/baseontologyActivity> ;\n    rdfs:label "Leucocytes"'},
 {'corpus_id': 1,
  'score': np.float32(-3.039883),
  'text': 'log:concept%3Aname_IV%20Antibiotics a <http://infs.cit.tum.de/karibdis/baseontologyActivity> ;\n    rdfs:label "IV Antibiotics"'},
 {'corpus_id': 5,
  'score': np.float32(-3.1868439),
  'text': 'log:concept%3Aname_CRP a <http://infs.cit.tum.de/karibdis/baseontologyActivity> ;\n    rdfs:label "CRP"'},
 {'corpus_id': 0,
  'score': np.float32(-3.2420154),
  'text': 'log:concept%3Aname_ER%20Sepsis%20Triage a <http://infs.cit.tum.de/karibdis/baseontologyActivity> ;\n    rdfs:label "ER Sepsis Triage"'},
 {'corpus_id': 11,
  'score': np.float32(-3.299514),
  'text': 'log:Diagnose_RC a log:type_Diagnose ;\n    rdfs:label "RC"'}]

In [130]:
results[URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count')]

[{'corpus_id': 1,
  'score': np.float32(0.7488426),
  'text': 'log:concept%3Aname_Leucocytes a <http://infs.cit.tum.de/karibdis/baseontologyActivity> ;\n    rdfs:label "Leucocytes"'},
 {'corpus_id': 2,
  'score': np.float32(0.48716342),
  'text': 'log:Diagnose_LE a log:type_Diagnose ;\n    rdfs:label "LE"'},
 {'corpus_id': 0,
  'score': np.float32(-0.29920658),
  'text': 'log:processValue_Leucocytes a :ProcessValue ;\n    rdfs:comment "The number of white blood cells in a blood test."'},
 {'corpus_id': 13,
  'score': np.float32(-0.45431447),
  'text': 'log:Diagnose_K a log:type_Diagnose ;\n    rdfs:label "K"'},
 {'corpus_id': 15,
  'score': np.float32(-0.48457402),
  'text': 'log:Diagnose_N a log:type_Diagnose ;\n    rdfs:label "N"'}]

In [104]:
def match_addition(addition_graph, target_graph):

    matches = dict()

    # addition_texts = textualize_graph(addition_texts)
    # target_texts = textualize_graph(target_graph)

    target_keys = list(target_texts.keys())
    target_docs = [target_texts[key] for key in target_keys]
    for node_to_add, add_text in addition_texts.items():
        # clear_output(wait=True)
        print(f'{len(matches)}/{len(addition_texts)}', end='\r')
        
        ranks = model.rank(add_text, target_docs, top_k=5, return_documents=True)

        if False:
            print('\n=======\n\n' + node_to_add)
            for i, rank in enumerate(ranks):
                print(f"\n{i+1}. (Score {rank['score']:.2f})\t{target_keys[rank['corpus_id']]}\n{rank['text']}")
        
            response = input(f'Type in [1-{len(ranks)}] to create matching for {node_to_add}. Type anything else to not create a matching\n')
            response_value = -1
            try:
                response_value = int(response)
            except ValueError:
                pass
            if response_value in range(1, len(ranks)+1):
                matched = target_keys[ranks[response_value-1]['corpus_id']]
                print(f'Matched with {matched.split("#")[-1]}')
                matches.append((node_to_add, matched))
            else:
                print('Skip')
        else: 
            matches[node_to_add] = ranks
    return matches

match_addition(sepon, pkg)

KeyboardInterrupt: 

In [ ]:
process_nodes = set(pkg.subjects(RDF.type, object=importer.entity_type_node('process_value')))

matches = []

for process_node in process_nodes:
    
    # clear_output(wait=True)
    
    process_node_label = process_node.split('/')[-1]
    ranks = model.rank(process_node_label, contexts)[:3]

    print('\n=======\n\n' + process_node_label)
    for i, rank in enumerate(ranks):
        print(f"\n{i+1}. (Score {rank['score']:.2f})\t{context_ids[rank['corpus_id']]}\n{contexts[rank['corpus_id']]}")

    response = input(f'Type in [1-{len(ranks)}] to create matching for {process_node_label}. Type anything else to not create a matching\n')
    response_value = -1
    try:
        response_value = int(response)
    except ValueError:
        pass
    if response_value in range(1, len(ranks)+1):
        matched = context_ids[ranks[response_value-1]['corpus_id']]
        print(f'Matched with {matched.split("#")[-1]}')
        matches.append((process_node, matched))
    else:
        print('Skip')



DiagnosticLacticAcid

1. (Score -7.45)	http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Endotoxin
ns1:Endotoxin rdfs:label "Endotoxin" ;
    ns1:Synonyms """Bacterial Pyrogen
Endotoxin
Endotoxins""" ;
    ns1:definition "The lipopolysaccharide complexes that are part of the outer membrane of the cell wall of Gram-negative bacteria such as E. coli, Salmonella, Shigella, Pseudomonas, Neisseria, Haemophilus, and other leading pathogens. Upon bacterial infections, the lipid component (Lipid A) of endotoxin contributes to its toxicity, while the polysaccharide components contribute to immunogenicity." ;
    rdfs:subClassOf ns1:Paediatric_Sepsis_Diagnostic_Biomarker

2. (Score -7.65)	http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Monocyte_Differentiation_Antigen_CD14
ns1:Monocyte_Differentiation_Antigen_CD14 rdfs:label "Monocyte Differentiation Antigen CD14" ;
    ns1:Synonyms """CD14 Antigen
Monocyte Differentiation Antigen CD14
Lipopolysaccharide R

Type in [1-3] to create matching for DiagnosticLacticAcid. Type anything else to not create a matching
 


Skip


Oligurie

1. (Score -9.00)	http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Optineurin
ns1:Optineurin rdfs:label "Optineurin" ;
    ns1:Synonyms """Huntingtin Interacting Protein L
Huntingtin-Interacting Protein 7
Transcription Factor IIIA-Interacting Protein
TFIIIA-IntP
Optic Neuropathy-Inducing Protein
Tumor Necrosis Factor Alpha-Inducible Cellular Protein Containing Leucine Zipper Domains
E3-14.7K-Interacting Protein
Huntingtin Yeast Partner L
Huntingtin-Interacting Protein L
NEMO-Related Protein
FIP-2
HIP-7
Optineurin
OPTN""" ;
    ns1:definition "Optineurin (577 aa, ~66 kDa) is encoded by the human OPTN gene. This protein plays a role in the maintenance of both neurons and the Golgi apparatus." ;
    rdfs:subClassOf ns1:Adult_Sepsis_Diagnostic_Biomarker,
        ns1:Regulatory_Protein

2. (Score -9.23)	http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Midkine
ns1:Midkine rdfs:label "Midkine" ;
    ns1:Synonyms """Neurite Outgrowth-Promot

Type in [1-3] to create matching for Oligurie. Type anything else to not create a matching
 


Skip


CRP

1. (Score 4.87)	http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein
ns1:C-Reactive_Protein rdfs:label "C-Reactive Protein" ;
    ns1:Synonyms """C-Reactive Protein
Pentraxin 1
C-Reactive Protein [Precursor]
Pentraxin 1, Short
CRP""" ;
    ns1:definition "C-reactive protein (224 aa, ~25 kDa) is encoded by the human CRP gene. This protein is cleaved during biological activation and is associated with host defense mechanisms and inflammatory responses." ;
    rdfs:subClassOf ns1:Adult_Sepsis_Diagnostic_Biomarker,
        ns1:Adult_Sepsis_Prognostic_Biomarker,
        ns1:Immunoprotein,
        ns1:Neonatal_Sepsis_Biomarker,
        ns1:Neonatal_Sepsis_Prognostic_Biomarker,
        ns1:Paediatric_Sepsis_Diagnostic_Biomarker,
        ns1:Paediatric_Sepsis_Prognostic_Biomarker

2. (Score 2.39)	http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C_Reactive_Protein_to_Mean_Platelet_Volume_Ratio
ns1:C_Reactive_Protein_to_Mean_Platel

...

Skip


Leucocytes

1. (Score -4.00)	http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Granulocyte_Count
ns1:Granulocyte_Count rdfs:label "Granulocyte Count" ;
    ns1:Synonyms """Granulocytes
Granulocyte Count
Polymorphonuclear Leukocytes
GRAN""" ;
    ns1:definition "The determination of the amount of granulocytes present in a sample." ;
    rdfs:subClassOf ns1:Leukocyte_Count

2. (Score -4.01)	http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count
ns1:Leukocyte_Count rdfs:label "Leukocyte Count" ;
    ns1:Synonyms """White Blood Cell Count
White Cell Count
White Blood Cells
Leukocyte Count
Leukocytes
WBC""" ;
    ns1:definition "A test to determine the number of leukocytes in a biospecimen." ;
    rdfs:subClassOf ns1:Blood_Cell_Count,
        ns1:Neonatal_Sepsis_Biomarker,
        ns1:Paediatric_Sepsis_Diagnostic_Biomarker

3. (Score -4.12)	http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leucine-Rich_Alpha-2-Glycoprot

Type in [1-3] to create matching for Leucocytes. Type anything else to not create a matching
 2


Matched with Leukocyte_Count


Age

1. (Score -7.54)	http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Acute_Physiology_and_Chronic_Health_Evaluation_II_Clinical_Classification
ns1:Acute_Physiology_and_Chronic_Health_Evaluation_II_Clinical_Classification rdfs:label "Acute Physiology and Chronic Health Evaluation II Clinical Classification" ;
    ns1:Synonyms """Acute Physiology and Chronic Health Evaluation II Clinical Classification
APACHE II
APCH1""" ;
    ns1:definition "A standardized rating scale developed by Knaus et al in 1985, which is a classification system used to measure severity of disease. This instrument uses a point score system based on physiological measurements, age and previous health status." ;
    rdfs:subClassOf ns1:Adult_Sepsis_Diagnostic_Biomarker,
        ns1:Adult_Sepsis_Prognostic_Biomarker,
        ns1:Clinical_Assessment_Indicator

2. (Score -8.82)	http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#20S_Proteasome
ns1:20S_

Type in [1-3] to create matching for Age. Type anything else to not create a matching
 


Skip


# ===============================================================

### Extract "Rules"

In [37]:
from pm4py import discover_declare, discover_log_skeleton

In [38]:
declare = discover_declare(log, min_support_ratio=0.98, min_confidence_ratio=0.98)

In [39]:
declare

{'responded_existence': {('ER Registration',
   'ER Sepsis Triage'): {'support': 1050, 'confidence': 1049},
  ('ER Sepsis Triage', 'ER Registration'): {'support': 1049,
   'confidence': 1049},
  ('ER Registration', 'ER Triage'): {'support': 1050, 'confidence': 1050},
  ('ER Sepsis Triage', 'ER Triage'): {'support': 1049, 'confidence': 1049},
  ('ER Triage', 'ER Sepsis Triage'): {'support': 1050, 'confidence': 1049},
  ('ER Triage', 'ER Registration'): {'support': 1050, 'confidence': 1050}},
 'existence': {'ER Sepsis Triage': {'support': 1050, 'confidence': 1049},
  'ER Registration': {'support': 1050, 'confidence': 1050},
  'ER Triage': {'support': 1050, 'confidence': 1050}},
 'exactly_one': {'ER Sepsis Triage': {'support': 1049, 'confidence': 1049},
  'ER Registration': {'support': 1050, 'confidence': 1050},
  'ER Triage': {'support': 1050, 'confidence': 1047}},
 'altprecedence': {('ER Registration', 'ER Sepsis Triage'): {'support': 1049,
   'confidence': 1042},
  ('ER Registration', 

In [155]:
discover_log_skeleton(log)

{'equivalence': {('ER Sepsis Triage', 'ER Registration'),
  ('IV Antibiotics', 'ER Registration'),
  ('IV Antibiotics', 'ER Sepsis Triage'),
  ('IV Liquid', 'ER Registration'),
  ('IV Liquid', 'ER Sepsis Triage'),
  ('IV Liquid', 'IV Antibiotics'),
  ('Release A', 'ER Registration'),
  ('Release B', 'ER Registration'),
  ('Release B', 'ER Sepsis Triage'),
  ('Release B', 'ER Triage'),
  ('Release C', 'ER Registration'),
  ('Release C', 'ER Sepsis Triage'),
  ('Release C', 'ER Triage'),
  ('Release D', 'ER Registration'),
  ('Release D', 'ER Sepsis Triage'),
  ('Release D', 'ER Triage'),
  ('Release E', 'ER Registration'),
  ('Release E', 'ER Sepsis Triage'),
  ('Release E', 'ER Triage'),
  ('Return ER', 'ER Registration')},
 'always_after': {('Release B', 'Admission NC'),
  ('Release C', 'Return ER'),
  ('Release D', 'Return ER'),
  ('Release E', 'Return ER')},
 'always_before': set(),
 'never_together': {('Release A', 'Release B'),
  ('Release A', 'Release C'),
  ('Release A', 'Releas

# Ontology Extraction

In [7]:
from src.utils import *

In [9]:
def nodes_in_dist(graph, nodes, dist, filter_func=lambda x: not x.startswith(OWL) and not x.startswith(RDF) and not x.startswith(RDFS)):
    triples = set()
    for node in nodes:
        triples |= set(graph.triples((node, None, None)))
        triples |= set(graph.triples((None, None, node)))
    if dist > 1:
        neighborhood = set([x[0] for x in triples]) | set([x[2] for x in triples]) 
        neighborhood = set(filter(filter_func, neighborhood)) 
        print(neighborhood)
        triples |= nodes_in_dist(graph, neighborhood, dist-1)
    return triples

In [10]:
URIRef('http://www.w3.org/2002/07/owl#Class').startswith(OWL)

True

In [11]:
OWL

Namespace('http://www.w3.org/2002/07/owl#')

In [12]:
lc_node = URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count')
g2 = Graph()
copy_namespaces(g2, sepon)
g2 += nodes_in_dist(sepon, [lc_node], 2)
#g2 += list(sepon.triples((x, Path(), None)))
#g2 += list(sepon.triples((None, Path(), x)))
draw_graph(g2)

{rdflib.term.Literal('http://ncicb.nci.nih.gov/xml/owl/EVS/Thesaurus.owl#C51948'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Neonatal_Sepsis_Biomarker'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Blood_Cell_Count'), rdflib.term.Literal('Leukocyte Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Granulocyte_Count'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Neutrophil_Count'), rdflib.term.Literal('White Blood Cell Count\nWhite Cell Count\nWhite Blood Cells\nLeukocyte Count\nLeukocytes\nWBC'), rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Eosinophil_Count'), rdflib.term.Literal('A test to determine the number of leukocytes in a biospecimen.'), rdflib.term.URI

GraphWidget(layout=Layout(height='800px', width='100%'))

In [13]:
x = URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Diagnostic_Procedure')
g2 = Graph()
copy_namespaces(g2, sepon)
g2 += nodes_in_dist(sepon, sepon.subjects(object=x, predicate=RDFS.subClassOf * '*'), 1)
#g2 += list(sepon.triples((x, Path(), None)))
#g2 += list(sepon.triples((None, Path(), x)))
# draw_graph(g2)
list()

[]

In [14]:
len(g2)

3215

In [15]:
len(sepon)

12942

In [20]:
log.dtypes

InfectionSuspected                        object
org:group                                 object
DiagnosticBlood                           object
DisfuncOrg                                object
SIRSCritTachypnea                         object
Hypotensie                                object
SIRSCritHeartRate                         object
Infusion                                  object
DiagnosticArtAstrup                       object
concept:name                              object
Age                                      float64
DiagnosticIC                              object
DiagnosticSputum                          object
DiagnosticLiquor                          object
DiagnosticOther                           object
SIRSCriteria2OrMore                       object
DiagnosticXthorax                         object
SIRSCritTemperature                       object
time:timestamp               datetime64[ns, UTC]
DiagnosticUrinaryCulture                  object
SIRSCritLeucos      

In [22]:
from rdflib import OWL

In [23]:
from rdflib import Literal
Literal(5).datatype

rdflib.term.URIRef('http://www.w3.org/2001/XMLSchema#integer')

rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#label')

In [249]:
pkg.namespace['foo']

rdflib.term.URIRef('http://example.org/foo')

### Matching With Ontology

In [26]:
list(pkg.subjects(RDF.type, object=importer.entity_type_node('process_value')))[0].split('/')[-1]

'InfectionSuspected'

In [28]:
ontology_nodes = set(sepon.subjects(object=URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Diagnostic_Biomarker'), predicate=RDFS.subClassOf * '*'))
context_ids = []
contexts = []
for node in ontology_nodes:
    g = Graph()
    g += context(node, sepon)
    _context = '\n\n'.join(g.serialize(format='ttl').split('.\n\n')[1:]).strip() # Remove namespaces
    context_ids.append(node)
    contexts.append(_context)
    # display(list(context(node, sepon)))

[print(f'\n====\n\n{x}\n{y}') for x,y in zip(context_ids, contexts)]


====

http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hyaluronan_Synthases
ns1:Hyaluronan_Synthases rdfs:label "Hyaluronan Synthases" ;
    ns1:Synonyms """Synthase 3, Hyaluronan
Synthetase, Hyaluronic Acid
Synthetase, Hyaluronan
HAS1 Protein
Synthase 2, Hyaluronan
Synthase, Hyaluronate
Hyaluronan Synthase 3
Hyaluronate Synthetase
Synthase 1, Hyaluronan
HAS2 Protein
Protein, HAS3
Hyaluronan Synthase
Hyaluronic Acid Synthetase
HAS3 Protein
3, Hyaluronan Synthase
Hyaluronan Synthetase
Hyaluronan Synthase 2
Synthase, Hyaluronan
Synthetase, Hyaluronate
Hyaluronate Synthase
hasA Enzyme
Synthases, Hyaluronan
Hyaluronan Synthase 1""" ;
    ns1:definition "Membrane-associated glucuronosyltransferases that catalyze the reaction of UDP-N-acetyl-D-glucosamine and UDP-D-glucuronate to produce HYALURONAN. HYALURONAN SYNTHASE 2 (HAS2) is essential for embryogenesis and its expression by tumor cells is associated with metastasis." ;
    rdfs:subClassOf ns1:Adult_Sepsis_Diagnostic

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

SIRSCritHeartRate -> sepon:Heart_Rate is not a match, one is the parameter, the other is a boolean on that parameter

In [30]:
matches

[(rdflib.term.URIRef('http://example.org/instances/process_values/CRP'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein')),
 (rdflib.term.URIRef('http://example.org/instances/process_values/LacticAcid'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate')),
 (rdflib.term.URIRef('http://example.org/instances/process_values/Leucocytes'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count'))]

In [60]:
_matches = [(URIRef('http://example.org/instances/process_values/CRP'),
  URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein')),
 (URIRef('http://example.org/instances/process_values/LacticAcid'),
  URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate')),
 (URIRef('http://example.org/instances/process_values/Leucocytes'),
  URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count'))]

In [61]:
manually_added_matchings = [
    (importer.entity_instance_node('process_value', 'Hypoxie'), URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia'))    
]

In [65]:
matches.extend(manually_added_matchings)
matches

[(rdflib.term.URIRef('http://example.org/instances/process_values/CRP'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein')),
 (rdflib.term.URIRef('http://example.org/instances/process_values/LacticAcid'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate')),
 (rdflib.term.URIRef('http://example.org/instances/process_values/Leucocytes'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count')),
 (rdflib.term.URIRef('http://example.org/instances/process_values/Hypoxie'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia'))]

In [66]:
def rename_identifier(g: Graph, old_uri: URIRef, new_uri: URIRef):
    # Collect all affected triples
    to_add = []
    to_remove = []
    
    for s, p, o in g:
        if s == old_uri or o == old_uri:
            new_s = new_uri if s == old_uri else s
            new_o = new_uri if o == old_uri else o
            to_remove.append((s, p, o))
            to_add.append((new_s, p, new_o))
    
    # Apply changes
    for triple in to_remove:
        g.remove(triple)
    for triple in to_add:
        g.add(triple)

In [70]:
for old, new in matches:
    rename_identifier(pkg, old, new)
    pkg += context(new, sepon)
    print(f'Replaced {old}\nwith {new}')

Replaced http://example.org/instances/process_values/CRP
with http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#C-Reactive_Protein
Replaced http://example.org/instances/process_values/LacticAcid
with http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Lactate
Replaced http://example.org/instances/process_values/Leucocytes
with http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Leukocyte_Count
Replaced http://example.org/instances/process_values/Hypoxie
with http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia


In [88]:
pkg.bind('sepon', 'http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#')

In [35]:
#TODO memorize this translation

In [69]:
list(context(URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia'), sepon))

[(rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia'),
  rdflib.term.URIRef('http://www.w3.org/2000/01/rdf-schema#subClassOf'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Ventilation_or_Perfusion_Signs_and_Symptoms')),
 (rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Synonyms'),
  rdflib.term.Literal('Hypoxia\nHypoxic\nhypoxia\nhypoxic')),
 (rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Hypoxia'),
  rdflib.term.URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#definition'),
  rdflib.term.Literal('A decrease in the amount of oxygen in the body. Symptoms range from mild (impaired judgment, memory loss, impaired motor coordination) to severe (seizures and coma).')),
 (rdflib.term.URIRef('htt

In [89]:
draw_graph(pkg)

GraphWidget(layout=Layout(height='800px', width='100%'))

#### Matching Workbench

from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')
biomarker_embeddings = model.encode(contexts, convert_to_tensor=True)
process_nodes = set(pkg.subjects(RDF.type, object=importer.entity_type_node('process_value')))
for process_node in process_nodes:
    process_node_label = process_node.split('/')[-1]
    name_embedding = model.encode(process_node_label, convert_to_tensor=True)
    
    # Compute cosine similarity with all biomarker contexts
    similarities = util.cos_sim(name_embedding, biomarker_embeddings)[0]
    
    # Get top 3 matches
    top_indices = similarities.argsort(descending=True)[:3]

    display(similarities)

process_nodes = set(pkg.subjects(RDF.type, object=importer.entity_type_node('process_value')))
ontology_nodes = set(sepon.subjects(object=URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Diagnostic_Biomarker'), predicate=RDFS.subClassOf * '*'))

from difflib import SequenceMatcher

matches = {}
for process_node in process_nodes:
    process_node_label = process_node.split('/')[-1]
    best_match = None
    best_score = 0
    for n2 in ontology_nodes:
        label2 = n2.split('#')[-1]
        score = SequenceMatcher(None, process_node_label, label2).ratio()
        if score > best_score:
            best_score = score
            best_match = (n2, label2)
    print(f"{process_node_label} -> {best_match[1]} (score: {best_score:.2f})")
    # confirm = input("Accept match? (y/n): ")
    #if confirm.lower() == 'y':
    if True:
        matches[process_node] = best_match[0]

matches

set(sepon.subjects(object=URIRef('http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#Diagnostic_Biomarker'), predicate=RDFS.subClassOf * '*'))

# Text Extraction

### Crawl the rules from the Clinical Guidelines

In [72]:
import requests
import re
from bs4 import BeautifulSoup

In [73]:
response = requests.get("https://www.sccm.org/clinical-resources/guidelines/guidelines/surviving-sepsis-guidelines-2021")
response.raise_for_status()
# print(response.text)

In [74]:
soup = BeautifulSoup(response.text, "html.parser")

In [75]:
guideline_container = soup.findAll("div", {"class": "col-main p-0"})[0]

In [76]:
reg = re.compile(r'\r\nQuality of evidence:.*')
rules = []
for guideline in guideline_container.findAll("div", {"class": "card mb-3"}):
    text = guideline.find('p', {'class' : 'mb-1'}).text
    text = reg.sub('', text)
    print(text)
    print('')
    rules.append(text)

For hospitals and health systems, we recommend using a performance improvement program for sepsis, including sepsis screening for acutely ill, high-risk patients and standard operating procedures for treatment.

We recommend against using qSOFA compared with SIRS, NEWS, or MEWS as a single screening tool for sepsis or septic shock.

For adults suspected of having sepsis, we suggest measuring blood lactate.

Sepsis and septic shock are medical emergencies, and we recommend that treatment and resuscitation begin immediately.

For patients with sepsis-induced hypoperfusion or septic shock, we suggest that at least 30 mL/kg of IV crystalloid fluid be given within the first 3 hours of resuscitation.

For adults with sepsis or septic shock, we suggest using dynamic measures to guide fluid resuscitation over physical examination or static parameters alone.

For adults with sepsis or septic shock, we suggest guiding resuscitation to decrease serum lactate in patients with elevated lactate leve

In [77]:
rules

['For hospitals and health systems, we recommend using a performance improvement program for sepsis, including sepsis screening for acutely ill, high-risk patients and standard operating procedures for treatment.',
 'We recommend against using qSOFA compared with SIRS, NEWS, or MEWS as a single screening tool for sepsis or septic shock.',
 'For adults suspected of having sepsis, we suggest measuring blood lactate.',
 'Sepsis and septic shock are medical emergencies, and we recommend that treatment and resuscitation begin immediately.',
 'For patients with sepsis-induced hypoperfusion or septic shock, we suggest that at least 30 mL/kg of IV crystalloid fluid be given within the first 3 hours of resuscitation.',
 'For adults with sepsis or septic shock, we suggest using dynamic measures to guide fluid resuscitation over physical examination or static parameters alone.',
 'For adults with sepsis or septic shock, we suggest guiding resuscitation to decrease serum\xa0lactate in patients wit

In [78]:
import json
cg_filename = './clinical_guidelines.json'

In [79]:
with open(cg_filename, 'w') as f:
    json.dump(rules, f)

In [80]:
with open(cg_filename) as f:
    assert rules == json.load(f)

In [81]:
with open(cg_filename) as f:
    rules = json.load(f)

### Alt: Curated Rules

In [82]:
curated_rules = [
    # From Surviving SEPSIS Guidelines
    'For adults with septic shock and severe metabolic acidemia (pH ≤ 7.2) and acute kidney injury (AKIN score 2 or 3), we suggest using sodium bicarbonate therapy.',
    'For adults with septic shock and hypoperfusion-induced lactic acidemia, we suggest against using sodium bicarbonate therapy to improve hemodynamics or to reduce vasopressor requirement',
    'For adults with sepsis or septic shock, we suggest against using IV vitamin C.',
    'For adults with sepsis or septic shock, we recommend initiating insulin therapy at a glucose level of ≥ 180mg/dL (10mmol/L).',
    'For adults with sepsis or septic shock, we recommend using low-molecular-weight heparin.',
    'For adults with sepsis or septic shock, we suggest against using IV immunoglobulin.',
    'For adults with septic shock and an ongoing requirement for vasopressor therapy, we suggest using IV corticosteroids.',
    'For adults with sepsis or septic shock who require ICU admission, we suggest admitting the patients to the ICU within 6 hours.',
    'For patients with sepsis-induced hypoperfusion or septic shock, we suggest that at least 30 mL/kg of IV crystalloid fluid be given within the first 3 hours of resuscitation.',
    # From Papers etc.
    '1. between ER Sepsis Triage and IV Antibiotics should be less than 1 hour',
    '2. between ER Sepsis Triage and LacticAcid should be less than 3 hours.'
    
]

### Translate the Rules

In [51]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [52]:
from dotenv import load_dotenv
load_dotenv()

True

In [53]:
llm = ChatOpenAI(temperature=0, model="gpt-4o-mini")

context_graph = Graph()
context_graph.parse('./src/base_ontology.ttl')
print(context_graph.serialize(format='ttl'))

In [55]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
'''Your task is to translate a textual rule into an equivalent SPARQL query based on a given ontology, so that whenever the rule is violated, the result set of the query is non-empty and vice versa. 
Create a query that includes the variable "?case", which relates to one specific process instance.
Please formulate a SELECT-WHERE query. 
Please only output the SPARQL query and nothing else. Don't output any notes or justifications.

For generating the rules, please consider the following contextual schema information in OWL-RDF format and key entities in RDF format.  

{context}''',
        ),
        ("human", "Please translate the following rule {rule}"),
    ]
)

prompt_context = context_graph.serialize(format='ttl') + '''

@prefix sepsis: <http://example.org/sepsis/> .

sepsis:Measurement rdf:type owl:Class ;
    rdfs:label "Measurement" ;    
    rdfs:comment "A measurement of a clinical parameter".

sepsis:Clinical_Parameter rdf:type owl:Class ;
    rdfs:label "Clinical Parameter" ;    
    rdfs:comment "A clinical parameter".

sepsis:measurand rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range sepsis:Clinical_Parameter .

sepsis:measurand rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range sepsis:Clinical_Parameter .

sepsis:mes_value rdf:type owl:DatatypeProperty, owl:FunctionalProperty ;    
    rdfs:comment "The value of a measurement" ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range xsd:decimal .

sepsis:produced rdf:type owl:ObjectProperty, owl:FunctionalProperty ;    
    rdfs:comment "A measurement has been produced" ;
    rdfs:domain type:Task ;
    rdfs:range sepsis:Measurement .

sepsis:leucocyte_count a sepsis:Clinical_Parameter ;
    rdfs:label "Leucocyte Count" ;    
    rdfs:comment "The number of Leucocytes (white blood cells) in 10^9 per liter of blood.".
'''

In [115]:
prompt_context = pkg.serialize(format='ttl')
print('\n'.join(prompt_context.split('\n')[:100]))

@prefix Activity: <http://example.org/instances/Activitys/> .
@prefix Diagnose: <http://example.org/instances/Diagnoses/> .
@prefix lifecycle%3Atransition: <http://example.org/instances/lifecycle%3Atransitions/> .
@prefix ns1: <http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#> .
@prefix org%3Agroup: <http://example.org/instances/org%3Agroups/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix process_value: <http://example.org/instances/process_values/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix relation: <http://example.org/relations/> .
@prefix type: <http://example.org/types/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

Activity:Admission%20IC a type:Activity .

Activity:Admission%20NC a type:Activity .

Activity:CRP a type:Activity ;
    relation:writes_value process_value:Age,
        ns1:C-Reactive_Protein .

Activity:ER%20Registration a type:Activity ;
    re

In [118]:
#TODO fix this bug, this is just a workaround
prompt_context = prompt_context.replace('ns1:', 'sepon:')

``` python
context = '''
@prefix relation: <http://example.org/relations/> .
@prefix type: <http://example.org/types/> .
@prefix sepsis: <http://example.org/sepsis/> .

type:Task rdf:type owl:Class ;
    rdfs:label "Task" ;
    rdfs:comment "One specific task to be performed".

type:Case rdf:type owl:Class ;
    rdfs:label "Case" ;
    rdfs:comment "One instance of a process such as one shopping order".

type:Activity rdf:type owl:Class ;
    rdfs:label "Activity" ;    
    rdfs:comment "A recurring activity of a process".

relation:instanceOf rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain type:Task ;
    rdfs:range type:Activity .

relation:partOf rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain type:Task ;
    rdfs:range type:Case .


sepsis:Measurement rdf:type owl:Class ;
    rdfs:label "Measurement" ;    
    rdfs:comment "A measurement of a clinical parameter".

sepsis:Clinical_Parameter rdf:type owl:Class ;
    rdfs:label "Clinical Parameter" ;    
    rdfs:comment "A clinical parameter".

sepsis:measurand rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range sepsis:Clinical_Parameter .

sepsis:measurand rdf:type owl:ObjectProperty, owl:FunctionalProperty ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range sepsis:Clinical_Parameter .

sepsis:mes_value rdf:type owl:DatatypeProperty, owl:FunctionalProperty ;    
    rdfs:comment "The value of a measurement" ;
    rdfs:domain sepsis:Measurement ;
    rdfs:range xsd:decimal .

sepsis:produced rdf:type owl:ObjectProperty, owl:FunctionalProperty ;    
    rdfs:comment "A measurement has been produced" ;
    rdfs:domain type:Task ;
    rdfs:range sepsis:Measurement .

sepsis:leucocyte_count a sepsis:Clinical_Parameter ;
    rdfs:label "Leucocyte Count" ;    
    rdfs:comment "The number of Leucocytes (white blood cells) in 10^9 per liter of blood.".
    
'''
```

In [119]:
chain = prompt | llm

def translate_rule(rule):
     return chain.invoke(
        {
            "context": prompt_context,
            "rule": rule,
        }
    )

rule = "Leucocyte measurements must be between 4.0 and 12.0 million."
response = translate_rule(rule)
print(response.content)

```sparql
SELECT ?case WHERE {
    ?task relation:partOf ?case .
    ?task relation:performedBy ?resource .
    ?task instanceOf Activity:Leucocytes .
    ?value a sepon:Leukocyte_Count ;
           relation:data_type xsd:double .
    FILTER(?value < 4.0 || ?value > 12.0)
}
```


In [122]:
print(unwrap_markdown_code(response.content).replace(' instanceOf', ' relation:instanceOf'))

SELECT ?case WHERE {
    ?task relation:partOf ?case .
    ?task relation:performedBy ?resource .
    ?task relation:instanceOf Activity:Leucocytes .
    ?value a sepon:Leukocyte_Count ;
           relation:data_type xsd:double .
    FILTER(?value < 4.0 || ?value > 12.0)
}


In [112]:
pkg.query('''
SELECT ?case WHERE {
  ?task relation:partOf ?case .
  ?task relation:performedBy ?resource .
  ?task relation:instanceOf Activity:Leucocytes .
  ?value a sepo:Leukocyte_Count ;
         relation:data_type ?leukocyteCount .
  FILTER(?leukocyteCount < 4.0 || ?leukocyteCount > 12.0)
}
''')

In [125]:
list(pkg.query(unwrap_markdown_code(response.content.replace(' instanceOf', ' relation:instanceOf'))))

[]

In [102]:
printmd('<span style="color:red">ERROR</span>')

<span style="color:red">ERROR</span>

In [126]:
for rule in curated_rules:
    print(rule)
    response = translate_rule(rule)
    printmd(response.content)
    try:
        res = list(pkg.query(unwrap_markdown_code(response.content.replace(' instanceOf', ' relation:instanceOf'))))
        print(res)
    except Exception as e:
        printmd('<span style="color:red">ERROR</span>')
        print(e)

For adults with septic shock and severe metabolic acidemia (pH ≤ 7.2) and acute kidney injury (AKIN score 2 or 3), we suggest using sodium bicarbonate therapy.


```sparql
SELECT ?case
WHERE {
  ?case a type:Case .
  ?task a type:Task ;
        relation:partOf ?case ;
        relation:performedBy ?resource .
  ?resource a type:Resource .
  ?task relation:instanceOf Activity:IV%20Liquid .
  ?task relation:writes_value process_value:Age .
  ?task relation:writes_value process_value:DiagnosticBlood .
  ?task relation:writes_value process_value:DiagnosticLacticAcid .
  ?task relation:writes_value process_value:DisfuncOrg .
  ?task relation:writes_value process_value:Hypotensie .
  ?task relation:writes_value process_value:InfectionSuspected .
  FILTER (?pH <= 7.2) .
  FILTER (?AKIN_score >= 2 && ?AKIN_score <= 3) .
}
```

[]
For adults with septic shock and hypoperfusion-induced lactic acidemia, we suggest against using sodium bicarbonate therapy to improve hemodynamics or to reduce vasopressor requirement


```sparql
SELECT ?case
WHERE {
  ?case a type:Case .
  ?task a type:Task ;
        relation:partOf ?case ;
        relation:performedBy ?resource .
  ?activity a type:Activity ;
            relation:instanceOf ?task .
  ?activity relation:writes_value sepon:Lactate ;
            relation:writes_value sepon:Hypoxia .
  FILTER NOT EXISTS {
    ?activity relation:writes_value process_value:IV%20Liquid .
  }
}
```

[]
For adults with sepsis or septic shock, we suggest against using IV vitamin C.


...

[]
1. between ER Sepsis Triage and IV Antibiotics should be less than 1 hour


```sparql
SELECT ?case
WHERE {
  ?task1 a type:Task ;
          relation:partOf ?case ;
          instanceOf Activity:ER%20Sepsis%20Triage ;
          relation:completedAt ?time1 .
  ?task2 a type:Task ;
          relation:partOf ?case ;
          instanceOf Activity:IV%20Antibiotics ;
          relation:completedAt ?time2 .
  FILTER(?time2 - ?time1 >= xsd:duration("PT1H"))
}
```

[]
2. between ER Sepsis Triage and LacticAcid should be less than 3 hours.


```sparql
SELECT ?case
WHERE {
  ?task1 a type:Task ;
         relation:instanceOf Activity:ER%20Sepsis%20Triage ;
         relation:partOf ?case ;
         relation:completedAt ?time1 .
  ?task2 a type:Task ;
         relation:instanceOf Activity:LacticAcid ;
         relation:partOf ?case ;
         relation:completedAt ?time2 .
  FILTER(?time2 - ?time1 >= "PT3H"^^xsd:duration)
}
```

[]


# Getting it all in some kind of UI

In [100]:
import ipywidgets
from IPython.display import display

In [101]:
text = ipywidgets.Text(description='What\'s your name?')
button = ipywidgets.Button(description='Rename', on_click=lambda b: print(text.value))
output = ipywidgets.Output()

def on_button_click(b):
    with output:
        print(text.value)

button.on_click(on_button_click)
display(text, button, output)

Text(value='', description="What's your name?")

Button(description='Rename', style=ButtonStyle())

Output()

# Lab

In [170]:
prompt2 = ChatPromptTemplate.from_messages(
    [
        (
            "system",
'''You are a knowledge importer for a knowledge-graph-based business process management system.
Your task is to input textual process knowledge and an existing knowledge graph and output new nodes and edges representing the knowledge of the text.
Reuse existing owl:classes and owl:properties where appropriate. 

Please output the nodes and edges in RDF-turtle-syntax.
Output nothing else. Don't output any notes or justifications.

For generating the nodes and edges, please consider the following contextual schema information in OWL-RDF format and key entities in RDF format.  

{context}''',
        ),
        ("human", "{rule}"),
    ]
)
chain2 = prompt2 | llm

In [171]:
text = 'Draw at least two sets of blood cultures (aerobic and anaerobic) and cultures from suspected sources (e.g., urine, sputum, wound) before antibiotic administration. However, never delay antibiotics for culture collection beyond the initial hour.'
text = 'The activity CRP describes that a blood test for measuring CRP values is performed'
# text = 'Leucocytes, Lactic acid and CRP are all measurements provided by blood tests. Leucocytes are white blood cells and a high white blood count can signal an infection the body may be fighting [7]. CRP, or C-reactive protein can signal inflammation in the liver if the test result is high [8].'
response = chain2.invoke(
    {
        "context": prompt_context,
        "rule": text,
    }
).content
printmd(response)

```turtle
Activity:CRP a type:Activity ;
    rdfs:comment "A blood test for measuring CRP values is performed." .
```

'Activity:CRP a type:Activity ;\n    rdfs:comment "A blood test for measuring CRP values is performed." .'

In [176]:
print(f'{namespace_string(pkg)}\n\n{unwrap_markdown_code(response)}')

@prefix Activity: <http://example.org/instances/Activitys/> .
@prefix Diagnose: <http://example.org/instances/Diagnoses/> .
@prefix lifecycle%3Atransition: <http://example.org/instances/lifecycle%3Atransitions/> .
@prefix ns1: <http://www.semanticweb.org/zchero/ontologies/2023/11/SepsisOntology#> .
@prefix org%3Agroup: <http://example.org/instances/org%3Agroups/> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix process_value: <http://example.org/instances/process_values/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix relation: <http://example.org/relations/> .
@prefix type: <http://example.org/types/> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

Activity:CRP a type:Activity ;
    rdfs:comment "A blood test for measuring CRP values is performed." .


In [197]:
from itertools import zip_longest

{'a': '#999900', 'b': '#999900', 'c': '#999900'}

In [192]:
dict(zip(addition.all_nodes() - pkg.all_nodes(), ['#999900']))

{rdflib.term.Literal('A blood test for measuring CRP values is performed.'): '#999900'}

In [213]:
pkg -= list(Graph().parse(data=f'{namespace_string(pkg)}\n\n{unwrap_markdown_code(response)}', format='turtle'))[1:]

In [217]:
addition = Graph().parse(data=f'{namespace_string(pkg)}\n\n{addition_code}', format='turtle')
addition -= pkg
draw_graph(addition, color_func=lambda _: dict(zip_longest(addition.all_nodes() - pkg.all_nodes(), [], fillvalue='#99AA00')))
text = ipywidgets.Textarea(description='Current Code', value=addition_code)
button = ipywidgets.Button(description='Reload', on_click=lambda b: print(text.value))
output = ipywidgets.Output()

def on_button_click(b):
    pass

button.on_click(on_button_click)

GraphWidget(layout=Layout(height='500px', width='100%'))

In [203]:
pkg += addition

In [235]:
from rdflib.namespace import Namespace, DefinedNamespace
from rdflib.term import URIRef

BASE_URL = 'http://infs.cit.tum.de/karibdis/base/'

class BASE_PROCESS_ONTOLOGY(DefinedNamespace):

    _fail = True

    relation : URIRef
    
    _NS = Namespace(BASE_URL)

In [236]:
BASE_PROCESS_ONTOLOGY.relation

rdflib.term.URIRef('http://infs.cit.tum.de/karibdis/base/relation')